# Problema Inverso Estacionário

Este notebook tem como objetivo implementar um modelo PINN para resolver um problema inverso estacionário.

**Autor**: Edélio Gabriel Magalhães de Jesus.

## O que são problemas inversos?

Como foi discutido no *notebook* introdutório (se ainda não leu, vale a pena conferir `00_introduction.ipynb`), as PINNs podem ser utilizadas para resolver problemas inversos. Mas você, muito provavelmente, pode estar se perguntando: o que são problemas inversos?

---

### Definição

No paradigma clássico da simulação numérica — o **problema direto** — conhecemos completamente a física do sistema: os parâmetros da equação, as condições de contorno e as condições iniciais. A partir disso, determinamos a solução $u(\mathbf{x})$. É o caminho das *causas* para os *efeitos*.

O **problema inverso** percorre o caminho oposto: a partir de observações da solução — medições esparsas, ruidosas ou indiretas — busca-se determinar alguma quantidade desconhecida do sistema. Na definição de Engl et al. [[ref]](#engl): *"resolver um problema inverso é determinar causas desconhecidas a partir de efeitos desejados ou observados"*.

<div style="text-align: center;">
  <img 
    src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTU7Uq083Cuy8c89AF6eMvLEHzUAbXIK489CQ&s" 
    alt="Problema direto vs. Problema Inverso"
    style="max-width: 400px; width: 50%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://downloads.editoracientifica.com.br/articles/220709394.pdf' target='_blank'>
    Problemas Inversos: Conceitos Gerais e Aplicação Prática
  </a>
</div>

A quantidade desconhecida pode assumir formas muito distintas dependendo do problema — um coeficiente da equação, um termo fonte, uma condição de contorno inacessível, ou até uma propriedade do material. O que unifica todos os casos é a estrutura: você possui *dados* e quer extrair deles *informação física* que não é diretamente observável.

---

Para conferir algumas aplicações reais, veja o *notebook* introdutório!

## O nosso problema

Como discutido anteriormente, problemas inversos buscam determinar causas desconhecidas a partir de efeitos observados. Nesse exemplo, trabalharemos com a **equação de Poisson-Boltzmann** — uma EDP não linear com aplicação direta em eletroquímica, ciência de coloides e caracterização de superfícies carregadas [[ref]](#pb-wiki).

---

### `Condições de contorno`

As condições de contorno são:

$$
\tilde{\psi}(0) = \tilde{\psi}_0
$$

$$
\tilde{\psi}(\infty) = 0
$$

Além disso, em implementações numéricas, não é possível trabalhar com um domínio infinito. Assim, truncamos o domínio em um valor suficientemente grande:

$$
\tilde{x} \in [0,L]
$$

onde escolhemos:

$$
L \approx 5
$$

pois após alguns comprimentos de Debye o potencial já é praticamente nulo. Além disso, vamos usar uma amplitudade de ruído gaussiano de 0.1.

---

### `Requisitos teóricos`

#### **Contexto físico**

Quando uma superfície eletricamente carregada é imersa em uma solução eletrolítica, os íons do líquido passam a se reorganizar próximos da superfície e, conforme a teoria clássica da Eletrostática:

- Íons de carga oposta são atraísdos;
- Íons de de msma carga são repelidos.

Esse rearranjo cria uma região próxima à superfície onde existe excesso de carga elétrica, formando a chamada **dupla camada elétrica** (muito conhecida pelo termo em inglês *double*).

Como consequência, surge um potencial elétrico $\psi(x)$ que:

- é maior próximo da superfície;
- decai gradualmente;
- tende a zero longe dela.

<div style="text-align: center;">
  <img 
    src="https://upload.wikimedia.org/wikipedia/commons/3/34/Gouy-Chapman_Simple_Model.jpg"
    alt="Dupla camada elétrica"
    style="max-width: 400px; width: 50%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://en.wikipedia.org/wiki/Poisson%E2%80%93Boltzmann_equation' target='_blank'>
    Wikipedia — Poisson-Boltzmann equation
  </a>
</div>

A distribuição do potencial elétrico pode ser modelada pela **Equação de Poisson-Boltxmann**, uma EDP não linear muito utilizada em :

- Eletroquímica;
- Ciencia de coloides;
- Biofísica Molecular;
- Interfaces carregadas.

Na forma dimensional, a equação é:

$$
\nabla^2\psi = -\frac{\\rho(\psi)}{\epsilon}
$$

onde:

- $\psi$ é o potencial elétrico;
- $\rho(\psi)$ é a densidade de carga iônica;
- $\varepsilon$ é a permisividade elétrica do meio.

Na teoria de campo médio, assume-se que os íons seguem uma distribuição de Boltzmann. Isso leca à forma:

$$\nabla^2\psi = \frac{e}{\varepsilon}\sum_i z_i c_i^0 \exp\left(-\frac{z_i e \psi}{k_B T}\right) \tag{1}$$

onde:

- $e$ é a carga elementar
- $z_i$ é a valência de íon $i$;
- $c_i^0$ é a concentração da espécie $i$ longe da superfície;
- $k_B$ é a constante de Boltzmann;
- $T$ é a temperatura.

A principal característica dessa equação é a sua não linearidade, causada pelo termo exponencial.

#### **Simplificação para geometria planar**

Considere agora uma superfície plana infinita.

Por simetria, o potencial varia apenas na direção perpendicular à superfície. Assim, o problema tridimensional reduz-se a uma EDO unidimensional:

$$
\frac{d^2\psi}{dx^2} = \frac{2ec_0}{\epsilon}sinh\left(\frac{e\psi}{k_BT}\right)
$$

Interpretando fisicamente, o que a fórmula nos diz é que o lado esquerdo mede a curvatura do potencial, enquanto que o direito a redistribuição dos íons no eletrólito.

#### **Adimensionalização**

Embora a equação anterior já descreva corretamente o fenômeno físico, ela ainda contém diversas constantes físicas carregando unidades. Isso dificulta tanto a análise matemática quanto a implementação numérica.

Para simplificar o problema, introduzimos duas variáveis adimensionais:

- o potencial adimensional

$$
\tilde{\psi} = \frac{e\psi}{k_B T}
$$

- e a coordenada espacial adimensional

$$
\tilde{x} = \kappa x
$$

onde $\kappa^{-1}$ é o chamado **comprimento de Debye**, a escala característica de decaimento do potencial elétrico na solução.

Fisicamente, o comprimento de Debye indica até que distância a influência elétrica da superfície é percebida no eletrólito:

- soluções mais concentradas possuem menor comprimento de Debye - devido ao fenômeno de blindagem eletrônica;
- portanto, o potencial decai mais rapidamente.

Substituindo essas variáveis na equação anterior, obtemos a forma adimensional clássica da equação de Poisson-Boltzmann:

$$
\frac{d^2\tilde{\psi}}{d\tilde{x}^2}
=
\sinh(\tilde{\psi})
\tag{2}
$$

Agora toda a física do problema encontra-se encapsulada na variável adimensional $\tilde{\psi}$.

A primeira condição representa o potencial elétrico na superfície carregada, enquanto a segunda expressa que o potencial tende a zero longe da interface.

---

#### **O problema inverso**

No problema inverso, faremos exatamente o caminho contrário.

Em vez de conhecer $\tilde{\psi}_0$ e calcular o perfil $\tilde{\psi}(\tilde{x})$, assumimos que possuímos medições experimentais ruidosas do potencial no interior do domínio.

O objetivo passa então a ser:

> recuperar o valor desconhecido de $\tilde{\psi}_0$ que melhor explica os dados observados.

Matematicamente:

- problema direto:

$$
\tilde{\psi}_0
\longrightarrow
\tilde{\psi}(\tilde{x})
$$

- problema inverso:

$$
\tilde{\psi}(\tilde{x})
\longrightarrow
\tilde{\psi}_0
$$

Esse não é um cenário totalmente impossível, quando pensamos em:

- caracterização eletroquímica;
- inferência de propriedades de superfícies;
- ajuste de parâmetros físicos;
- reconstrução baseada em dados experimentais.

Nesse exemplo, utilizaremos medições sintéticas ruidosas do potencial elétrico para estimar o valor desconhecido de $\tilde{\psi}_0$.


## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex04_pinn_inverse_stationary.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_stationary
from plot_utils import plot_loss, plot_heatmaps, plot_points_transient, plot_profiles, plot_pb, plot_psi0_evolution
from ex04_pinn_inverse_stationary import analytical_solution_pb, generate_synthetic_data_pb, get_boundary_points_pb, train_pb, evaluate_pb

### **Parâmetros do problema**

Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados na discussão presente artigo original de "Raissi et. al. ("**Data-driven solutions of nonlinear partial differential equations**"[[ref]](#original-paper)), apenas para ter uma base.

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS = 1
N_OUTPUTS = 1
N_HIDDEN = 16
N_LAYERS = 4
ACTIVATION = nn.Tanh

# Parâmetros do problema
PSI0_TRUE  = 2.0
L          = 5.0

# Parâmetros de amostragem
N_COLLOC   = 1000
N_OBS      = 40
NOISE_AMP  = 0.1

# Parâmetros para o treinamento
N_EPOCHS   = 10000
LR         = 1e-3
W_PDE      = 1.0
W_BC       = 1.0
W_DATA     = 1.0

Usando: cpu


### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION).to(DEVICE)
psi0  = nn.Parameter(torch.tensor([1.0], dtype=torch.float32, device=DEVICE))

> ### OBSERVE QUE...

...aqui surge a principal diferença entre uma PINN para um problema direto e uma PINN para um problema inverso.

No problema direto, todos os parâmetros físicos da equação são conhecidos, e a rede neural aprende apenas a função solução $\psi(x)$.

Já no problema inverso, parte da física é desconhecida. Nesse caso, além dos pesos e vieses da rede neural, também queremos estimar um parâmetro físico do sistema — aqui, o potencial de superfície $\psi_0$.

Isso é feito através da instrução:

---

```python
psi0 = nn.Parameter(torch.tensor([1.0], dtype=torch.float32, device=DEVICE))
```

---

### **Amostragem dos pontos**

In [5]:
X_COLLOC        = sample_collocation_rectangular(N_COLLOC, [0], [L], DEVICE)
X_OBS, PSI_OBS  = generate_synthetic_data_pb(PSI0_TRUE, L, N_OBS, NOISE_AMP, DEVICE)
X_BC = torch.tensor([[0.0], [L]], dtype=torch.float32, device=DEVICE)

### **Instanciando o otimizador**

In [6]:
optimizer = torch.optim.Adam(
    list(model.parameters()) + [psi0],
    lr=LR
)

### **Treinamento**

In [7]:
history = train_pb(
    model, psi0, optimizer,
    X_COLLOC, X_BC, X_OBS, PSI_OBS,
    N_EPOCHS, W_PDE, W_BC, W_DATA
)

Epoch 00000 | Loss: 1.22e+00 | Loss PDE: 9.66e-03 | Loss BC: 6.83e-01 | Loss data: 5.31e-01 | psi0: 0.9990
Epoch 00100 | Loss: 1.77e-01 | Loss PDE: 7.22e-02 | Loss BC: 7.03e-03 | Loss data: 9.80e-02 | psi0: 0.9354
Epoch 00200 | Loss: 8.26e-02 | Loss PDE: 6.96e-03 | Loss BC: 9.59e-03 | Loss data: 6.60e-02 | psi0: 0.9448
Epoch 00300 | Loss: 7.59e-02 | Loss PDE: 5.33e-03 | Loss BC: 9.83e-03 | Loss data: 6.07e-02 | psi0: 0.9835
Epoch 00400 | Loss: 6.93e-02 | Loss PDE: 4.55e-03 | Loss BC: 9.12e-03 | Loss data: 5.56e-02 | psi0: 1.0282
Epoch 00500 | Loss: 6.26e-02 | Loss PDE: 3.96e-03 | Loss BC: 8.22e-03 | Loss data: 5.05e-02 | psi0: 1.0764
Epoch 00600 | Loss: 5.62e-02 | Loss PDE: 3.44e-03 | Loss BC: 7.29e-03 | Loss data: 4.55e-02 | psi0: 1.1266
Epoch 00700 | Loss: 5.01e-02 | Loss PDE: 2.98e-03 | Loss BC: 6.37e-03 | Loss data: 4.07e-02 | psi0: 1.1777
Epoch 00800 | Loss: 4.44e-02 | Loss PDE: 2.56e-03 | Loss BC: 5.51e-03 | Loss data: 3.63e-02 | psi0: 1.2290
Epoch 00900 | Loss: 3.91e-02 | Loss P

### **Visualizando os resultados do treinamento**

Nossa função de perda ficou da seguinte forma:

$$
Loss = \omega_{PDE} * L_{PDE} + \omega_{BC} * L_{BC} + \omega_{DATA} *  L_{DATA}
$$

Note que, aqui, temos algo interessante: os dados observados. Como estamos tratando de um problema inverso, então consideramos que tivermos acesso aos dados do nosso sistema, e a partir deles - junto do resíduo da equação diferencial e das condições de contorno - consiquiremos recuperar parâmetros da prórpria equação física que o descreve.

No código temos:

---
```python
[...]
    # perda física
    residual = pde_residual_pb(model, X_col)
    loss_pde = torch.mean(residual ** 2)

    # perda de contorno
    Psi_bc_true = torch.cat([psi0.view(1, 1),
                             torch.zeros(1, 1, device=X_bc.device)], dim=0)
    Psi_bc_pred = model(X_bc)
    loss_bc     = torch.mean((Psi_bc_pred - Psi_bc_true) ** 2)

    # perda de dados
    Psi_obs_pred = model(X_obs)
    loss_data    = torch.mean((Psi_obs_pred - Psi_obs) ** 2)

    loss = w_pde * loss_pde + w_bc * loss_bc + w_data * loss_data

    return loss, loss_pde, loss_bc, loss_data
```
---
    

In [8]:
plot_loss(history)
plot_psi0_evolution(history, PSI0_TRUE)

Observe que, mesmo com muitas flutuações nas perdas, em média, todas convergiram para valores muito bons, com a perda total ficando na ordem de grandeza de $10^{-3}$.

---


### **Validando o modelo**

Para validação, usamos a solução analítica.

---

#### **Solução analítica de Gouy-Chapman**

Para a geometria planar, existe uma solução analítica clássica conhecida como solução de Gouy-Chapman [[ref]](#pb-wiki):

$$
\tilde{\psi}(\tilde{x})
=
2\ln\left(
\frac{1 + Ce^{-\tilde{x}}}
{1 - Ce^{-\tilde{x}}}
\right)
\tag{3}
$$

onde

$$
C = \tanh\left(\frac{\tilde{\psi}_0}{4}\right)
$$

Observe que toda a solução é determinada por um único parâmetro:

$$
\tilde{\psi}_0
$$

que representa o potencial elétrico adimensional na superfície.

Ou seja, conhecendo $\tilde{\psi}_0$, conseguimos reconstruir todo o perfil de potencial no domínio - o problema direto!

No código, ela foi implementada na função `analitycal_solution_pb` simplesmente com:

---

```python
    C = np.tanh(psi0 / 4)
    return 2 * np.log((1 + C * np.exp(-x)) / (1 - C * np.exp(-x)))
```

---

depois chamada internanmente na função de validação `evaluate_pb`.

In [9]:
results = evaluate_pb(model, psi0, PSI0_TRUE, L, DEVICE)

In [10]:
plot_pb(results, X_OBS, PSI_OBS)

Como é possível observar, a PINN conseguiu recuperar muito bem o nosso parâmetro ($\psi$ na interface), mesmo com dados muito ruidosos.

Note, no entanto, que esse é um exemplo didático. Seria possível obter resultados semelhantes e até melhores com métodos clássicos - como mínimos quadrado - ou implementações modernas em *python* - como o scipy. 

> Então, por que usamos PINN????

Ora, voltemos às discussões do *notebook* introdutório (se ainda não conferiu, é absolutamente recomendável): imagine cenários que dificultam a possibilidade de adquirir muitos dados, e os poucos que você possui são absurdamente bons. Em contrapartida, você conhece a física que descreve esse cenário, e, mesmo sem uma solução analítica exata, como no nosso caso, seria possível recuperar o compotarmento (no caso de um problema direto) ou mesmo parâmetros da própria formulação física/matemática (como nese exemplo de problema inverso)!

Portanto, vale a pena investigar cada vez mais sobre como implementar PINNs nas mais diversas aplicações.

## Referências

<a id="livro-prob-inver"></a>  
IRILAN, Yves Garnard. *Problemas Inversos: Conceitos gerais e aplicação prática*. In: EDUCAÇÃO E ENSINO DE CIÊNCIAS E MATEMÁTICA: PESQUISA, APLICAÇÃO E NOVAS TENDÊNCIAS – VOLUME 2. Editora Científica Digital, 2022. p. 143–161. Disponível em: https://downloads.editoracientifica.com.br/articles/220709394.pdf

<a id="pb-wiki"></a>  
WIKIPEDIA. *Poisson–Boltzmann equation*. Disponível em: https://en.wikipedia.org/wiki/Poisson%E2%80%93Boltzmann_equation

<a id="prob-inverso"></a>  
WIKIPÉDIA. *Problema inverso*. Disponível em: https://pt.wikipedia.org/wiki/Problema_inverso